# Social Media Trend Analysis using PySpark 🚀

Analyze social media posts to find:
- 📈 **Trending hashtags**
- ❤️ **Engagement patterns** (likes, retweets)
- 😊 **Basic sentiment** (positive / negative)

**Tech Stack:** PySpark, Spark SQL, Parquet, Python

## ⚙️ Step 1: Setup Environment

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, when, date_format, sum as _sum, count, desc

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("SocialMediaTrendAnalysis") \
    .getOrCreate()

spark

## 📥 Step 2: Load Data
Upload your `social_media_posts.csv` file before running this step.

In [11]:
# Load CSV Data
df = spark.read.option("header", True).csv("social_media_posts.csv")

# Show initial data
print("Raw Data Preview:")
df.show(5)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/social_media_posts.csv. SQLSTATE: 42K03

## 🧹 Step 3: Data Cleaning & Preprocessing (ETL)

In [ ]:
# Convert numeric columns from string to int
df_clean = df.withColumn("likes", col("likes").cast("int")) \
             .withColumn("retweets", col("retweets").cast("int"))

# Print Schema to verify types
df_clean.printSchema()

## 📊 Step 4: Trending Hashtag Analysis
We split the `hashtags` column (pipe-separated) into individual rows to count occurrences.

In [ ]:
# Explode hashtags to get one row per hashtag
hashtags_df = df_clean.withColumn("hashtag", explode(split(col("hashtags"), "\\|")))

# Group by hashtag and count
trending = hashtags_df.groupBy("hashtag") \
    .count() \
    .orderBy(desc("count"))

print("🔥 Top 10 Trending Hashtags:")
trending.show(10)

## ❤️ Step 5: Engagement Analysis
Analyze total likes and retweets per day.

In [ ]:
# extract date from timestamp
engagement = df_clean.withColumn("date", split(col("timestamp"), " ").getItem(0))

# Aggregate engagement metrics by date
daily_engagement = engagement.groupBy("date") \
    .agg(_sum("likes").alias("total_likes"), 
         _sum("retweets").alias("total_retweets")) \
    .orderBy("date")

print("📅 Daily Engagement Metrics:")
daily_engagement.show(10)

## 😊 Step 6: Basic Sentiment Analysis
Rule-based classification using keywords.

In [ ]:
# Define sentiment rules using regex
df_sentiment = df_clean.withColumn(
    "sentiment",
    when(col("text").rlike("(?i)love|amazing|great|best|wonderful|excellent"), "positive")
    .when(col("text").rlike("(?i)hate|bad|worst|terrible|awful"), "negative")
    .otherwise("neutral")
)

print("🧠 Sentiment Analysis Results:")
df_sentiment.select("text", "sentiment").show(10, truncate=60)

## 💾 Step 7: Save Results (Parquet)
Save the processed data for future use.

In [ ]:
# Save trending hashtags
trending.write.mode("overwrite").parquet("output/trending_hashtags")

# Save processed data with sentiment
df_sentiment.write.mode("overwrite").parquet("output/processed_posts")

print("✅ Data saved to output/ directory")

## 📈 Step 8: Visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert top 10 trending to Pandas for plotting
pdf = trending.limit(10).toPandas()

plt.figure(figsize=(12, 6))
plt.bar(pdf['hashtag'], pdf['count'], color='skyblue')
plt.title('Top 10 Trending Hashtags')
plt.xlabel('Hashtag')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()